# A transformer that no longer fits

The model trains on one GPU. A larger version does not fit. Eight GPUs give us more
memory in total, but no layer can read another GPU's memory as if it were local.
We need to decide which weights and tokens each worker owns, then supply the missing
pieces through communication.

We will use the structure you already know:

```text
tokens → embeddings → [norm → attention → residual → norm → MLP → residual] × layers → logits
```

We will distribute **individual parts of this transformer**, one at a time: first a
training step with replicas, then the MLP, attention heads, experts, and sequence
positions. A different piece exposes a different way to divide the work. Each exercise
must reproduce that piece's ordinary PyTorch result; assembling a complete distributed
transformer comes later.

An MoE block replaces the dense MLP with selected experts. Normalization here means
LayerNorm or RMSNorm over each token's features; it does not mix sequence positions.
We assume you know forward, backward, attention heads, and a training step.

The large numbers below are memory and traffic calculations, not allocations. The
executable examples use tiny CPU tensors and Flock. They preserve the ownership and
numerical result of the operation, not GPU timing. DP includes a local backward and
optimizer update; the TP, EP, and CP puzzles check **forward only**. Correct forward
communication alone does not establish a correct distributed backward.

In [1]:
import copy

import flock
import flock.torch as ft
import torch

from dist_puzzles import check
from dist_puzzles.lessons.parallelisms import (
    check_columnwise_tp,
    check_ddp_step,
    check_expert_parallel,
    check_gathered_attention,
    check_ring_attention,
    check_rowwise_tp,
    check_tp_attention,
    check_tp_mlp,
    check_ulysses,
)

## 1. More examples, the same model

First keep the model small enough to fit. Give each rank an identical model and a
different minibatch. Local forward and backward need no messages. Before updating,
the ranks must agree on gradients; otherwise we train different models.

For equally sized batches and mean loss, $g=(g_0+\cdots+g_{P-1})/P$. This is **data
parallelism (DP)**. Here `ddp_step(model, x, target, learning_rate)` performs one SGD
step with mean squared error and returns a tuple of updated parameter copies.
The model and batch are supplied; the interesting part is between backward and step.

Predict the effect of omitting the division by $P$. Then run this worked example.

In [2]:
async def ddp_step(model, x, target, learning_rate):
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    optimizer.zero_grad()
    torch.nn.functional.mse_loss(model(x), target).backward()

    for parameter in model.parameters():
        gradient = await ft.all_reduce(parameter.grad, "sum").wait()
        parameter.grad = gradient / flock.get_world_size()

    optimizer.step()
    return tuple(parameter.detach().clone() for parameter in model.parameters())

Two replicas start with the scalar model $y=wx$, $w=0$. One sees `(x=1, target=2)`,
the other `(x=2, target=4)`. Their gradients are −4 and −16; the mean is −10.
With learning rate 0.1, both should finish with $w=1$.

In [3]:
model = torch.nn.Linear(1, 1, bias=False)
model.weight = torch.nn.Parameter(torch.zeros(1, 1))
inputs = [torch.tensor([[1.0]]), torch.tensor([[2.0]])]
targets = [2 * x for x in inputs]

run = flock.distribute(workers=2, seed=0)(ddp_step)
results = run(
    flock.per_rank([copy.deepcopy(model) for _ in inputs]),
    flock.per_rank(inputs),
    flock.per_rank(targets),
    learning_rate=0.1,
)
check(results, [(torch.tensor([[1.0]]),)] * 2)
results

[(tensor([[1.]]),), (tensor([[1.]]),)]

In [4]:
check_ddp_step(ddp_step)

All checks passed.


The checker compares two consecutive updates against training on the concatenated
batch, with several models, rank counts, and schedules. It also checks the actual
model parameters, not just the returned copies. Unequal local batch sizes would need
example-count weighting; this exercise uses equal sizes.

## 2. More devices did not make a replica smaller

Now increase the parameter count to a rounded **7 billion**. For this budget, choose
BF16 working weights and gradients, an FP32 master copy, and two FP32 Adam moments.
That is $2 + 2 + 4 + 4 + 4 = 16$ bytes per parameter. This is an explicit training
configuration, not a universal PyTorch allocation rule.

An [A100 80 GB](https://www.nvidia.com/en-us/data-center/a100/) gives a useful scale
for comparison. We use decimal GB for the device budget. The calculation excludes
saved activations, temporary communication buffers, workspaces, and allocator overhead.

In [5]:
parameter_count = 7_000_000_000
bytes_per_parameter = {
    "BF16 weights": 2,
    "BF16 gradients": 2,
    "FP32 master weights": 4,
    "FP32 Adam first moment": 4,
    "FP32 Adam second moment": 4,
}
state_bytes = parameter_count * sum(bytes_per_parameter.values())
for name, size in bytes_per_parameter.items():
    print(f"{name:24} {parameter_count * size / 1e9:6.1f} GB")
print(f"Total model state: {state_bytes / 1e9:.1f} GB")
for workers in (1, 2, 4, 8):
    print(
        f"{workers} workers: DP {state_bytes / 1e9:.1f} GB/rank; "
        f"ideal sharded state {state_bytes / workers / 1e9:.1f} GB/rank"
    )
assert state_bytes > 80e9

BF16 weights               14.0 GB
BF16 gradients             14.0 GB
FP32 master weights        28.0 GB
FP32 Adam first moment     28.0 GB
FP32 Adam second moment    28.0 GB
Total model state: 112.0 GB
1 workers: DP 112.0 GB/rank; ideal sharded state 112.0 GB/rank
2 workers: DP 112.0 GB/rank; ideal sharded state 56.0 GB/rank
4 workers: DP 112.0 GB/rank; ideal sharded state 28.0 GB/rank
8 workers: DP 112.0 GB/rank; ideal sharded state 14.0 GB/rank


DP still keeps 112 GB of model state on each rank. Splitting examples did not split
that state. Do we have to keep **every** training tensor replicated?

### ZeRO removes replication in stages

[ZeRO](https://www.deepspeed.ai/tutorials/zero/) progressively shards optimizer state,
then gradients, then working parameters. In our chosen precision scheme, the FP32
master weights and Adam moments together account for 12 of the 16 bytes per parameter.

| Strategy | Sharded state | Working weights between layers |
| --- | --- | --- |
| DP | None | Replicated |
| ZeRO-1 | Master weights and optimizer moments | Replicated |
| ZeRO-2 | The above, plus reduced gradients | Replicated |
| ZeRO-3 | The above, plus working weights | Sharded |

Owners update their parameter portions; replicas must receive the updated portions.
ZeRO-1/2 keep working weights available for layer computation, avoiding the per-layer
parameter gathers required by ZeRO-3. If they save enough memory, that can be preferable.
ZeRO-3 saves more persistent state but must materialize parameters when used.

In [6]:
workers = 4
state_per_parameter = {
    "DP": 16,
    "ZeRO-1": 2 + 2 + 12 / workers,
    "ZeRO-2": 2 + (2 + 12) / workers,
    "ZeRO-3": 16 / workers,
}
for strategy, size in state_per_parameter.items():
    print(f"{strategy:6}: {parameter_count * size / 1e9:5.1f} GB of model state per rank")

DP    : 112.0 GB of model state per rank
ZeRO-1:  49.0 GB of model state per rank
ZeRO-2:  38.5 GB of model state per rank
ZeRO-3:  28.0 GB of model state per rank


At four ranks, these budgets are 112, 49, 38.5, and 28 GB. ZeRO-1 already removes
63 GB of redundant state per rank in this example. More sharding is useful when the
remaining memory requirement demands it, not simply because the stage number is larger.
These are ideal model-state budgets, not peaks: temporary gradients, communication
buffers, active parameters, and saved activations still need room.

### FSDP gathers parameters when a layer needs them

Fully sharded data parallelism uses the full-sharding idea also present in ZeRO-3.
Before a module runs, its parameter shards are all-gathered. Each rank then computes
with full parameters on its own examples.

![Parameter shards followed by temporary full parameters on each rank](figures/fsdp-parameter-gather.png)

With resharding after forward, full parameters are freed and gathered again for backward.
Gradients are reduce-scattered to the owners, which update their state shards. See the
[PyTorch FSDP explanation](https://pytorch.org/blog/efficient-large-scale-training-with-pytorch/).
Gathering the whole model at once would defeat much of the memory saving; the unit of
gathering matters.

We will not implement FSDP here. Its useful question for the next puzzle is: if moving
large weights is expensive, can we compute with their shards and move activations instead?

## 3. Compute with the weights where they are

For an MLP, flatten batch and sequence into $T$ tokens. Let $X$ have shape $[T,D]$.
Our first linear layer has $W_0:[D,F]$ and bias $b_0:[F]$.

Split the **output features** of $W_0$ across ranks. Each rank receives the same $X$
and computes a finished slice of $XW_0+b_0$. These slices belong next to one another:
concatenating them restores the full output. This is column-wise tensor parallelism
(TP), using the mathematical $XW$ convention; `nn.Linear` stores transposed weights.

The worked example returns a full output to make the equivalence visible. `ft.all_gather`
returns a list in rank order; choose the concatenation dimension yourself.

In [7]:
async def columnwise_tp(X: torch.Tensor, W: torch.Tensor, b: torch.Tensor):
    part = X @ W + b[None, :]

    full = torch.cat(await ft.all_gather(part).wait(), dim=-1)

    return full

In [8]:
x = torch.tensor([[1.0, 2.0]])
w = torch.tensor([[1.0, 0.0, 2.0, 0.0], [0.0, 1.0, 0.0, 2.0]])
b = torch.zeros(4)

run = flock.distribute(workers=2, seed=0)(columnwise_tp)
results = run(x, flock.per_rank(w.chunk(2, dim=1)), flock.per_rank(b.chunk(2)))
check(results, [torch.tensor([[1.0, 2.0, 2.0, 4.0]])] * 2)
results

[tensor([[1., 2., 2., 4.]]), tensor([[1., 2., 2., 4.]])]

In [9]:
check_columnwise_tp(columnwise_tp)

All checks passed.


### A slice of the answer, or a contribution to it?

There is another way to split a linear layer. If rank $r$ has input features $X_r$
and the matching rows $W_r$, then $X_rW_r$ has the **full output shape**, but includes
only some terms of each dot product:

$$XW = \sum_r X_rW_r.$$

These outputs must be added, not concatenated. The replicated bias belongs in the
answer once, not once per rank. `ft.all_reduce(part, "sum")` returns the summed tensor.
Run this second worked example before joining the two layers.

In [10]:
async def rowwise_tp(X: torch.Tensor, W: torch.Tensor, b: torch.Tensor):
    part = X @ W
    full = await ft.all_reduce(part, "sum").wait()
    return full + b[None, :]

In [11]:
x = torch.tensor([[1.0, 2.0, 3.0, 4.0]])
w = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [0.0, 1.0]])
b = torch.tensor([10.0, 20.0])

run = flock.distribute(workers=2, seed=0)(rowwise_tp)
results = run(flock.per_rank(x.chunk(2, dim=1)), flock.per_rank(w.chunk(2)), b)
check(results, [torch.tensor([[14.0, 26.0]])] * 2)
results

[tensor([[14., 26.]]), tensor([[14., 26.]])]

In [12]:
check_rowwise_tp(rowwise_tp)

All checks passed.


### Puzzle 1 — leave the hidden activation split

The first example gathered its output. The second immediately needed separate input
features. Can the MLP avoid that intermediate gather?

Compute $Y=\operatorname{gelu}(XW_0+b_0)W_1+b_1$. Every rank starts with the same
$X:[T,D]$ and $b_1:[D]$. Rank $r$ owns columns $rF/P:(r+1)F/P$ of $W_0$, the matching
$b_0$, and those same rows of $W_1:[F,D]$. Return full $Y:[T,D]$ on every rank.

Keep the hidden activation $[T,F/P]$ local: do not gather hidden values or weights.
All sharded dimensions divide evenly. The checker compares against the full MLP on several shapes and schedules.

<details><summary>Hint: the operation between the layers</summary>
GELU acts independently on each hidden feature. It does not need the missing features.
</details>

In [13]:
async def col_row_mlp(
    X: torch.Tensor,
    W0: torch.Tensor,
    W1: torch.Tensor,
    b0: torch.Tensor,
    b1: torch.Tensor,
):
    hidden = torch.nn.functional.gelu(X @ W0 + b0[None, :])
    part = hidden @ W1
    full = await ft.all_reduce(part).wait()
    return full + b1[None, :]

In [14]:
x = torch.tensor([[1.0, -1.0], [0.0, 2.0]])
w0 = torch.tensor([[1.0, 0.0, 2.0, 0.0], [0.0, 1.0, 0.0, 2.0]])
w1 = w0.T.contiguous()
b0, b1 = torch.zeros(4), torch.tensor([1.0, 2.0])

run = flock.distribute(workers=2, seed=0)(col_row_mlp)
results = run(
    x, flock.per_rank(w0.chunk(2, dim=1)), flock.per_rank(w1.chunk(2)), flock.per_rank(b0.chunk(2)), b1
)
check(results, [torch.nn.functional.gelu(x @ w0 + b0) @ w1 + b1] * 2, rtol=1e-5, atol=1e-5)
results[0]

tensor([[ 5.7503,  1.7503],
        [ 1.0000, 11.9542]])

In [15]:
check_tp_mlp(col_row_mlp)

All checks passed.


### What did we save, and what did we start sending?

Ignoring biases, this two-matrix MLP has $2DF$ parameters. Each TP rank keeps $2DF/P$.
Its two matmuls also do about $1/P$ of the original work for the same tokens.

Use $D=4096$, $F=14336$ from the
[Mistral 7B configuration](https://huggingface.co/mistralai/Mistral-7B-v0.1/blob/main/config.json).
Our GELU MLP has **two** matrices; Mistral's gated MLP has three, and its attention uses
GQA and a sliding window. We borrow dimensions, not its full architecture.

Compare just forward traffic for four ranks, BF16 payloads, and this same two-matrix
block. A parameter all-gather sends $(P-1)/P$ of the full weight bytes per rank. A ring
all-reduce sends $2(P-1)/P$ of its activation tensor per rank. These are ideal algorithmic
byte counts, excluding biases, backward, latency, and overlap. FSDP ranks process separate
examples while TP ranks cooperate on the same tokens; this is not a throughput benchmark.

In [16]:
D, F, P = 4096, 14336, 4
weight_bytes = 2 * D * F * 2
print(f"Full two-matrix weights: {weight_bytes / 2**20:.1f} MiB")
print(f"TP weights per rank:    {weight_bytes / P / 2**20:.1f} MiB")
print("Tokens   weight gather sent/rank   activation all-reduce sent/rank")
for tokens in (512, 2048, 8192, 32768):
    gathered = (P - 1) / P * weight_bytes
    reduced = 2 * (P - 1) / P * tokens * D * 2
    print(f"{tokens:6} {gathered / 2**20:24.1f} MiB {reduced / 2**20:28.1f} MiB")

Full two-matrix weights: 224.0 MiB
TP weights per rank:    56.0 MiB
Tokens   weight gather sent/rank   activation all-reduce sent/rank
   512                    168.0 MiB                          6.0 MiB
  2048                    168.0 MiB                         24.0 MiB
  8192                    168.0 MiB                         96.0 MiB
 32768                    168.0 MiB                        384.0 MiB


The weight traffic stays fixed; the activation traffic grows with tokens. In this
comparison they cross at $T=F$. TP changes what travels; it does not make communication
free. FSDP and TP can also operate on different groups together.

### Puzzle 2 — split attention heads

Different attention heads can compute independently before the output projection.
Give each rank $H/P$ complete heads: columns of $W_Q,W_K,W_V:[D,D]$ and the matching
rows of $W_O:[D,D]$. The input $x:[B,S,D]$ is replicated, $D=H D_h$, and $H$ divides
evenly across ranks. We use bias-free projections to keep the interface small.

Implement `tensor_parallel_attention(x, wq, wk, wv, wo, num_heads)`. Return the full
$[B,S,D]$ output on every rank. Each worker receives only its local weight shards.
Compute **bidirectional** attention without dropout; combine partial output-projection
results, without gathering weights, Q/K/V, or local heads.

TP divides attention compute even without CP. The two attention matmuls cost about
$4BS^2D$ FLOPs in total, or $4BS^2D/P$ per rank (counting multiply and add separately).
The total work is unchanged.

In [17]:
async def tensor_parallel_attention(x, wq, wk, wv, wo, num_heads):
    batch, sequence, width = x.shape
    head_dim = width // num_heads
    local_heads = num_heads // flock.get_world_size()

    def project(weight):
        return (x @ weight).reshape(batch, sequence, local_heads, head_dim).transpose(1, 2)

    q, k, v = project(wq), project(wk), project(wv)
    local = (torch.softmax(q @ k.mT / head_dim**0.5, dim=-1) @ v).transpose(1, 2)
    partial = local.reshape(batch, sequence, -1) @ wo
    return await ft.all_reduce(partial, "sum").wait()

In [18]:
x = torch.tensor([[[1.0, 0.0], [0.0, 1.0]]])
w = torch.eye(2)
columns, rows = flock.per_rank(w.chunk(2, dim=1)), flock.per_rank(w.chunk(2))

run = flock.distribute(workers=2, seed=0)(tensor_parallel_attention)
results = run(x, columns, columns, columns, rows, num_heads=2)
results[0]

tensor([[[0.7311, 0.5000],
         [0.5000, 0.7311]]])

In [19]:
check_tp_attention(tensor_parallel_attention)

All checks passed.


## 4. A token does not need every expert

A dense MLP applies the same weights to every token. An MoE layer selects a few experts
for each token. That gives us another ownership choice: keep whole experts on different
ranks and send tokens to the owner of the selected expert. This is **expert parallelism
(EP)**. TP could still split the computation inside each expert.

For four experts on two ranks, IDs 0–1 live on rank 0 and IDs 2–3 on rank 1. A source
with assignments `[3, 0, 3]` sends two tokens to rank 1 and keeps one local. Returned
outputs must go back to positions `[0, 2]` and `[1]`, respectively.

### Puzzle 3 — dispatch and return

Rank $r$ owns only its contiguous $E/P$ experts and starts with $T_r$ tokens
$X_r:[T_r,D]$ and one expert ID per token. Our small expert is
$\operatorname{gelu}(xW_e+b_e)$; the supplied IDs replace a learned router.
Return the expert outputs in the original local token order.

Route tokens and routing information with all-to-all or point-to-point messages; keep expert weights local. Local token counts
may differ, including zero; a peer may have nothing for you. `ft.all_to_all(parts)`
takes one tensor per destination and returns one tensor per source, in rank order.
Pieces may have different first dimensions, including zero. Complete every exchange.
You choose how to keep enough information to restore order.

In [20]:
async def expert_parallel(
    X: torch.Tensor,
    expert_ids: torch.Tensor,
    expert_weights: torch.Tensor,
    expert_biases: torch.Tensor,
):
    rank = flock.get_rank()
    ws = flock.get_world_size()
    experts_per_rank = expert_weights.shape[0]
    owners = expert_ids // experts_per_rank
    send_indices = [torch.where(owners == dst)[0] for dst in range(ws)]
    received_tokens = await ft.all_to_all([X[indices] for indices in send_indices]).wait()
    received_ids = await ft.all_to_all([expert_ids[indices] for indices in send_indices]).wait()

    expert_start = rank * experts_per_rank
    output_sends = []
    for tokens, ids in zip(received_tokens, received_ids, strict=True):
        local_ids = ids - expert_start
        output = torch.einsum("td,tde->te", tokens, expert_weights[local_ids])
        output_sends.append(torch.nn.functional.gelu(output + expert_biases[local_ids]))

    received_outputs = await ft.all_to_all(output_sends).wait()

    Y = torch.empty_like(X)
    for indices, outputs in zip(send_indices, received_outputs, strict=True):
        Y[indices] = outputs
    return Y

In [21]:
tokens = [torch.tensor([[1.0, 0.0], [0.0, 2.0]]), torch.tensor([[3.0, 1.0]])]
assignments = [torch.tensor([1, 0]), torch.tensor([1])]
weights = [torch.eye(2).unsqueeze(0), (2 * torch.eye(2)).unsqueeze(0)]
biases = [torch.zeros(1, 2), torch.zeros(1, 2)]

run = flock.distribute(workers=2, seed=0)(expert_parallel)
results = run(
    flock.per_rank(tokens), flock.per_rank(assignments), flock.per_rank(weights), flock.per_rank(biases)
)
results

[tensor([[1.9545, 0.0000],
        [0.0000, 1.9545]]),
 tensor([[6.0000, 1.9545]])]

In [22]:
check_expert_parallel(expert_parallel)

All checks passed.


With top-1 routing, at most one input vector and one output vector travel per token;
local routes need no network transfer. With top-$k$, up to $k$ expert paths need to be
combined. For $T$ source tokens of width $D$, the remote dispatch+return payload is at
most $2kTD$ elements, excluding metadata and routing weights.

Splitting experts does not guarantee balanced compute. Count assignments to each owner
below: the same weight placement can receive very different loads. A larger EP group
can aggregate more tokens per expert under some batch layouts, but it does not guarantee
that every expert gets enough work.

In [23]:
for ids in ([0, 1, 2, 3, 0, 1, 2, 3], [0, 0, 0, 0, 0, 0, 2, 3]):
    owners = torch.tensor(ids) // 2
    print(f"assignments={ids} -> tokens per owner={torch.bincount(owners, minlength=2).tolist()}")

assignments=[0, 1, 2, 3, 0, 1, 2, 3] -> tokens per owner=[4, 4]
assignments=[0, 0, 0, 0, 0, 0, 2, 3] -> tokens per owner=[6, 2]


## 5. The weights fit; the context grows

Keep the model width, but increase the number of tokens in one sequence. Splitting
independent examples with DP cannot split this one example. Even token-wise activations
become large, before considering attention.

The next calculation uses BF16 storage, one sequence, and the same $D,F$ as above.
It counts individual tensors, **not total training memory**. The dense score matrix is
an illustration of naive attention: memory-efficient attention need not materialize it.
These long contexts are hypothetical dense-attention workloads, not supported-context
claims about Mistral 7B.

In [24]:
D, F, H = 4096, 14336, 32
print("Tokens   residual [S,D]   MLP hidden [S,F]   naive scores [H,S,S]")
for sequence in (8192, 32768, 262144):
    residual = sequence * D * 2
    hidden = sequence * F * 2
    scores = H * sequence * sequence * 2
    print(f"{sequence:6} {residual / 2**30:14.2f} GiB {hidden / 2**30:17.2f} GiB {scores / 2**30:22.1f} GiB")

Tokens   residual [S,D]   MLP hidden [S,F]   naive scores [H,S,S]
  8192           0.06 GiB              0.22 GiB                    4.0 GiB
 32768           0.25 GiB              0.88 GiB                   64.0 GiB
262144           2.00 GiB              7.00 GiB                 4096.0 GiB


**Context parallelism (CP)** assigns sequence positions to ranks cooperating on the
same example. Token-wise normalization and the MLP can process their positions locally.
Attention is different: a local query needs keys and values from other ranks.

For all three CP examples, the projections have already been computed. Each worker
receives `q, k, v`, each shaped **`[batch, heads, local_sequence, head_dim]`**, and returns
attention for its local query positions in the same layout. Shards are contiguous and
assigned in rank order. This lets us change the exchange without repeating projections
and reshapes. All attention here is **bidirectional (non-causal)**, without dropout.

### Start with the simplest correct exchange

Keep local queries and gather K/V. Since `ft.all_gather` returns a rank-ordered list,
concatenating on dimension 2 restores sequence order.

In [25]:
async def gathered_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    full_k = torch.cat(await ft.all_gather(k).wait(), dim=2)
    full_v = torch.cat(await ft.all_gather(v).wait(), dim=2)
    return torch.softmax(q @ full_k.mT / q.shape[-1] ** 0.5, dim=-1) @ full_v

In [26]:
queries = torch.arange(16, dtype=torch.float32).reshape(1, 2, 4, 2) / 8
keys = queries.flip(2)
values = torch.arange(16, dtype=torch.float32).reshape(1, 2, 4, 2)
inputs = [flock.per_rank(t.chunk(2, dim=2)) for t in (queries, keys, values)]
expected = torch.nn.functional.scaled_dot_product_attention(queries, keys, values)

run = flock.distribute(workers=2, seed=0)(gathered_attention)
results = run(*inputs)
check(results, list(expected.chunk(2, dim=2)), rtol=1e-5, atol=1e-5)
torch.cat(results, dim=2)

tensor([[[[ 2.9448,  3.9448],
          [ 2.7247,  3.7247],
          [ 2.5083,  3.5083],
          [ 2.2981,  3.2981]],

         [[10.0965, 11.0965],
          [ 9.9053, 10.9053],
          [ 9.7259, 10.7259],
          [ 9.5592, 10.5592]]]])

In [27]:
check_gathered_attention(gathered_attention)

All checks passed.


The output is right, but every rank temporarily holds full K/V: $2BSD$ elements for
ordinary multi-head attention. Can each worker instead own a complete attention problem
for just part of the heads?

### Puzzle 4 — Ulysses changes the sharding axis

Implement `ulysses(q, k, v)` with the same inputs and output as the baseline. During
attention, let each rank own **all sequence positions for $H/P$ heads**. Afterwards,
restore the original sequence ownership. This is the
[Ulysses](https://www.deepspeed.ai/tutorials/ds-sequence/) layout change.

Use all-to-all exchanges; do not gather all heads and positions on every rank. Both
sequence length and head count divide evenly across ranks. Work out which pieces each
destination needs and which axis joins the pieces from different sources. The list
returned by all-to-all is ordered by **source**, not destination.

In [28]:
async def ulysses(q, k, v):
    world = flock.get_world_size()

    async def full_sequence(tensor):
        pieces = await ft.all_to_all(tensor.chunk(world, dim=1)).wait()
        return torch.cat(pieces, dim=2)

    q, k, v = await full_sequence(q), await full_sequence(k), await full_sequence(v)
    output = torch.softmax(q @ k.mT / q.shape[-1] ** 0.5, dim=-1) @ v
    pieces = await ft.all_to_all(output.chunk(world, dim=2)).wait()
    return torch.cat(pieces, dim=1)

In [29]:
run = flock.distribute(workers=2, seed=0)(ulysses)
results = run(*inputs)
check(results, list(expected.chunk(2, dim=2)), rtol=1e-5, atol=1e-5)
torch.cat(results, dim=2)

tensor([[[[ 2.9448,  3.9448],
          [ 2.7247,  3.7247],
          [ 2.5083,  3.5083],
          [ 2.2981,  3.2981]],

         [[10.0965, 11.0965],
          [ 9.9053, 10.9053],
          [ 9.7259, 10.7259],
          [ 9.5592, 10.5592]]]])

In [30]:
check_ulysses(ulysses)

All checks passed.


During attention each rank now owns $BS(H/P)D_h$ elements of each Q/K/V tensor.
That is $1/P$ of each full tensor, while still providing all keys for its local heads.
Each equal-sized head receives the same complete attention problem, including the same
causal triangle if masking is added.

Our basic equal-head implementation needs $H$ divisible by $P$. With four heads, it
cannot use eight ranks this way. GQA/MQA require further care with the number of K/V
heads; this exercise uses ordinary multi-head attention. Could we keep sequence
ownership throughout attention instead?

## 6. Advanced — attention one K/V block at a time

[Ring Attention](https://arxiv.org/abs/2310.01889) keeps queries local and circulates
K/V blocks. It does not remove dense attention's quadratic total computation. It avoids
the need to keep all K/V on one rank and does not divide the heads across CP ranks.

There is one numerical obstacle before the communication puzzle. Softmax normalizes
across **all** keys. Averaging independently normalized block outputs is wrong: each
block has a different denominator.

For scores $A=QK_{block}^{T}/\sqrt{D_h}$, keep a running maximum $m$, exponential sum
$l$, and unnormalized weighted sum $O$, all per query and head. Start with
$m=-\infty$, $l=0$, $O=0$. For the next block set

$$m' = \max(m,\max A),\quad \alpha=\exp(m-m'),\quad P_b=\exp(A-m'),$$
$$l'=\alpha l+\sum P_b,\qquad O'=\alpha O+P_bV_{block}.$$

After all blocks, return $O/l$. Rescaling the old state puts old and new terms under
the same exponent reference. Try the following local example before distributing it:

In [31]:
def blockwise_attention(q: torch.Tensor, blocks: list[tuple[torch.Tensor, torch.Tensor]]) -> torch.Tensor:
    m = torch.full_like(q[..., :1], -torch.inf)
    denominator = torch.zeros_like(m)
    output = torch.zeros_like(q)
    for k, v in blocks:
        scores = q @ k.mT / q.shape[-1] ** 0.5
        next_m = torch.maximum(m, scores.max(dim=-1, keepdim=True).values)
        scale = torch.exp(m - next_m)
        probabilities = torch.exp(scores - next_m)
        denominator = scale * denominator + probabilities.sum(dim=-1, keepdim=True)
        output = scale * output + probabilities @ v
        m = next_m
    return output / denominator


q = torch.tensor([[[[1.0]]]])
k = torch.tensor([[[[0.0], [2.0]]]])
v = torch.tensor([[[[0.0], [10.0]]]])
block_expected = torch.softmax(q @ k.mT, dim=-1) @ v
actual = blockwise_attention(q, list(zip(k.chunk(2, dim=2), v.chunk(2, dim=2), strict=True)))
torch.testing.assert_close(actual, block_expected)
print(f"Mean of separate block outputs: 5.000; correct merged output: {actual.item():.3f}")

Mean of separate block outputs: 5.000; correct merged output: 8.808


### Puzzle 5 — move the blocks, keep the queries

Implement `ring_attention(q, k, v)` with the same sequence-sharded inputs and output as
the baseline. This is **bidirectional attention**: every local query must see every
K/V block. Head count need not divide the number of ranks.

Keep queries local. Process your K/V block and the other $P-1$ blocks received from
the left neighbour, sending only to the right neighbour. Do not gather full K/V or
save a list of received blocks. Besides the supplied inputs, retain only the current
block, at most one transfer buffer, and online softmax state. Complete all work before
returning; one rank needs no messages.

The online update above is available to you. The puzzle is to organize its inputs
without changing the normalization. Nonblocking calls expose an opportunity for
hardware overlap; Flock checks dependencies, not measured overlap.

In [32]:
async def ring_attention(q, k, v):
    rank, world = flock.get_rank(), flock.get_world_size()
    maximum = torch.full_like(q[..., :1], -torch.inf)
    denominator = torch.zeros_like(maximum)
    output = torch.zeros_like(q)

    for step in range(world):
        send = None
        if step < world - 1:
            send = ft.isend((rank + 1) % world, torch.stack((k, v)), tag=step)

        scores = q @ k.mT / q.shape[-1] ** 0.5
        next_maximum = torch.maximum(maximum, scores.max(dim=-1, keepdim=True).values)
        scale = torch.exp(maximum - next_maximum)
        probabilities = torch.exp(scores - next_maximum)
        denominator = scale * denominator + probabilities.sum(dim=-1, keepdim=True)
        output = scale * output + probabilities @ v
        maximum = next_maximum

        if send is not None:
            k, v = (await ft.recv((rank - 1) % world, tag=step)).unbind(0)
            await send.wait()

    return output / denominator

In [33]:
run = flock.distribute(workers=2, seed=0)(ring_attention)
results = run(*inputs)
check(results, list(expected.chunk(2, dim=2)), rtol=1e-5, atol=1e-5)
torch.cat(results, dim=2)

tensor([[[[ 2.9448,  3.9448],
          [ 2.7247,  3.7247],
          [ 2.5083,  3.5083],
          [ 2.2981,  3.2981]],

         [[10.0965, 11.0965],
          [ 9.9053, 10.9053],
          [ 9.7259, 10.7259],
          [ 9.5592, 10.5592]]]])

In [34]:
check_ring_attention(ring_attention)

All checks passed.


The checker includes large attention scores, different dtypes, and three heads on four
ranks. It compares with full bidirectional attention and checks ring links.

A causal mask opens another problem: equal query counts need not mean equal useful work.
More general masks make that balance harder still. We leave those algorithms outside
this exercise; [MagiAttention](https://sandai-org.github.io/MagiAttention/docs/main/index.html)
is a further reading direction for distributed attention with flexible masks and load balancing.

## Choosing what to split

| Constraint | Ownership choice | What crosses ranks here |
| --- | --- | --- |
| More independent examples | DP replicas | Gradients |
| Model state does not fit | FSDP state shards | Parameters and gradients |
| Compute a layer without gathering weights | TP weight and feature shards | Partial outputs |
| Only selected experts are needed | EP expert owners | Routed tokens and outputs |
| One long context does not fit | CP sequence shards | Attention inputs or K/V blocks |

These choices can coexist on different groups. The exercises isolate one choice so
that its numerical contract stays visible. Splitting complete **layers** creates a
different dependency: a stage waits for the preceding stage. Continue with
[Pipeline parallelism](05_pipeline_parallel.ipynb).

For larger-scale cost reasoning, see
[How To Scale Your Model](https://jax-ml.github.io/scaling-book/training/) and
[The Ultra-Scale Playbook](https://huggingface.co/spaces/nanotron/ultrascale-playbook).
Neither the byte calculations nor Flock runs here predict actual device throughput.